# 🎬 London Beer Flood 1814 - Ultra-Fast GPU Cloud Render

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arjunmalpani/london-beer-flood-1814/blob/main/render_colab.ipynb)

### ⚡ Instructions:
1. **Stop previous run**: Click the square Stop button (⏹) on any running cell.
2. Run all cells: click **Runtime** &rarr; **Run all** (or `Ctrl + F9`).
3. Hardware GPU acceleration (`chrome-for-testing` + NVIDIA NVENC) renders the entire video in **~1 to 2 minutes**!
4. The final 1080x1920 video `final_beer_flood_1814.mp4` will download automatically.

In [ ]:
# Step 1: Check NVIDIA GPU
!nvidia-smi

In [ ]:
# Step 2: Pull latest code & install dependencies
import os
if os.path.exists('london-beer-flood-1814'):
    %cd london-beer-flood-1814
    !git pull origin main
else:
    !git clone https://github.com/arjunmalpani/london-beer-flood-1814.git
    %cd london-beer-flood-1814

!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs ffmpeg xvfb libegl1-mesa-dev libgl1-mesa-dev libvulkan-dev
!npm install --legacy-peer-deps

In [ ]:
# Step 3: Verify Remotion GPU hardware acceleration
!npx remotion gpu --chrome-mode="chrome-for-testing" --gl=angle-egl

In [ ]:
# Step 4: Render with true NVIDIA GPU acceleration (1-2 minutes!)
# Using chrome-for-testing + angle-egl + hardware NVENC encoding
!mkdir -p out
!xvfb-run -a -s "-screen 0 1920x1080x24" npx remotion render src/index.ts BeerFlood out/raw_video.mp4 \
    --chrome-mode="chrome-for-testing" \
    --gl=angle-egl \
    --hardware-acceleration=if-possible \
    --scale=0.6667 \
    --concurrency=4 \
    --timeout=120000 \
    --overwrite

In [ ]:
# Step 5: Upscale to crisp 1080x1920 and mux master audio mix (-15 LUFS stereo)
!ffmpeg -y -i out/raw_video.mp4 -i public/audio/mix.wav \
    -vf "scale=1080:1920:flags=lanczos" \
    -map 0:v:0 -map 1:a:0 \
    -c:v libx264 -crf 18 -preset fast \
    -c:a aac -b:a 320k \
    -shortest \
    out/final_beer_flood_1814.mp4

!ls -lh out/final_beer_flood_1814.mp4
print("\n Render and Audio Muxing Completed Successfully!")

In [ ]:
# Step 6: Download the finished video
from google.colab import files
files.download('out/final_beer_flood_1814.mp4')